In [40]:
from ortools.sat.python import cp_model
import pandas as pd
import json
from tabulate import tabulate
import time
import sys

# --------------------
# Configuration Constants
# --------------------
BATCHES_PER_DIVISION = 4  # The class is divided into 4 batches

# --------------------
# Solution Callback Class
# --------------------
class TimetableSolutionPrinter(cp_model.CpSolverSolutionCallback):
    """Print intermediate solutions."""

    def __init__(self, limit):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self._solution_count = 0
        self._solution_limit = limit
        self._start_time = time.time()

    def on_solution_callback(self):
        current_time = time.time()
        print(f'Solution {self._solution_count} found in {current_time - self._start_time:.2f} seconds')
        self._solution_count += 1
        if self._solution_count >= self._solution_limit:
            print(f'Stopping search after finding {self._solution_limit} solutions.')
            self.StopSearch()

    def solution_count(self):
        return self._solution_count

def load_data_from_json():
    """Loads all necessary data from external JSON files."""
    try:
        with open('config.json', 'r') as f:
            config = json.load(f)
        with open('frequencies.json', 'r') as f:
            class_frequencies = json.load(f)
        with open('teachers.json', 'r') as f:
            teacher_assignments = json.load(f)
        with open('rooms.json', 'r') as f:
            room_assignments = json.load(f)
        
        print("Successfully loaded all JSON configuration files.")
        return config, class_frequencies, teacher_assignments, room_assignments
    except FileNotFoundError as e:
        print(f"Error: {e}. Make sure config.json, frequencies.json, teachers.json, and rooms.json are in the same directory.")
        sys.exit(1)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}. Please check the format of your JSON files.")
        sys.exit(1)


def main():
    # --------------------
    # Parameters from JSON
    # --------------------
    config, class_frequencies, teacher_assignments, room_assignments = load_data_from_json()

    divisions = config['divisions']
    lectures = config['lectures']
    labs = config['labs']
    days = config['days']
    
    # --- Dynamic Time Slot Calculation ---
    start_hour = config['start_hour']
    end_hour = config['end_hour']
    lunch_start_hour = config.get('lunch_start_hour')

    if start_hour >= end_hour:
        print("Error: start_hour must be less than end_hour in config.json")
        sys.exit(1)

    total_hours = end_hour - start_hour
    slots = list(range(total_hours))
    
    # Calculate lunch slot index dynamically
    LUNCH_SLOT = None
    if lunch_start_hour is not None and start_hour <= lunch_start_hour < end_hour:
        LUNCH_SLOT = lunch_start_hour - start_hour
        print(f"Lunch break is scheduled for slot index {LUNCH_SLOT} ({lunch_start_hour}:00 - {lunch_start_hour+1}:00).")
    else:
        print("No lunch break is scheduled within the specified college hours.")

    all_classes = lectures + labs
    num_classes = len(all_classes)
    class_to_idx = {cls: i for i, cls in enumerate(all_classes)}
    idx_to_class = {i: cls for cls, i in class_to_idx.items()}

    # --- Division-specific teacher mappings ---
    division_class_to_teacher = {}
    for d in divisions:
        division_class_to_teacher[d] = {}
        for teacher, classes in teacher_assignments.items():
            if teacher.startswith(f"{d}_"):
                for cls in classes:
                    division_class_to_teacher[d][cls] = teacher
    
    # Legacy mapping for backward compatibility and shared resources
    class_to_teacher = {cls: teacher for teacher, classes in teacher_assignments.items() for cls in classes}
    class_to_room = {cls: room for room, classes in room_assignments.items() for cls in classes}
    all_teachers = list(teacher_assignments.keys())
    all_rooms = list(room_assignments.keys())

    # Calculate scheduling feasibility
    available_slots_per_day = len(slots) - (1 if LUNCH_SLOT is not None else 0)
    total_weekly_slots = available_slots_per_day * len(days)
    
    # Calculate required slots
    lecture_hours_needed = sum(class_frequencies[lec] for lec in lectures)
    lab_hours_needed_per_batch = sum(class_frequencies[lab] * 2 for lab in labs)  # Labs are 2-hour sessions
    
    print(f"Planning timetable for {len(divisions)} divisions, each with {BATCHES_PER_DIVISION} batches")
    print(f"Available slots per day: {available_slots_per_day}, Total weekly slots: {total_weekly_slots}")
    print(f"Lectures: {lectures} (Total: {lecture_hours_needed} hours/week)")
    print(f"Labs: {labs} (Total per batch: {lab_hours_needed_per_batch} hours/week)")
    print(f"Note: Labs can accommodate 1/{BATCHES_PER_DIVISION} of class at a time")
    print(f"Teachers are now division-specific - each teacher teaches only one division")

    # --------------------
    # Model & Variables
    # --------------------
    model = cp_model.CpModel()
    
    # Variable Structure: x[division][batch][day][slot]
    x = {}
    
    for d in divisions:
        x[d] = {}
        for b in range(BATCHES_PER_DIVISION):
            x[d][b] = {}
            for day in days:
                x[d][b][day] = {}
                for s in slots:
                    # -1 means Free/No Class
                    x[d][b][day][s] = model.NewIntVar(-1, num_classes - 1, f'x_{d}_B{b}_{day}_{s}')

    # Add lunch break constraint
    if LUNCH_SLOT is not None:
        for d in divisions:
            for b in range(BATCHES_PER_DIVISION):
                for day in days:
                    model.Add(x[d][b][day][LUNCH_SLOT] == -1)

    # --------------------
    # LECTURE CONSTRAINTS - All batches attend together
    # --------------------
    print("Adding lecture constraints...")
    for d in divisions:
        for lec in lectures:
            lec_val = class_to_idx[lec]
            weekly_sessions = []
            
            for day in days:
                daily_sessions = []
                for s in slots:
                    if s == LUNCH_SLOT:
                        continue
                        
                    # Boolean variable for whether this lecture happens at this time
                    lec_happens = model.NewBoolVar(f'lec_{d}_{lec}_{day}_{s}')
                    daily_sessions.append(lec_happens)
                    
                    # If lecture happens, ALL batches must attend
                    for b in range(BATCHES_PER_DIVISION):
                        model.Add(x[d][b][day][s] == lec_val).OnlyEnforceIf(lec_happens)
                
                # At most one session of this lecture per day
                model.Add(sum(daily_sessions) <= 1)
                weekly_sessions.extend(daily_sessions)
            
            # Must have exactly the required frequency per week
            model.Add(sum(weekly_sessions) == class_frequencies[lec])

    # --------------------
    # LAB CONSTRAINTS - Independent batch scheduling
    # --------------------
    print("Adding lab constraints...")
    for d in divisions:
        for lab in labs:
            lab_val = class_to_idx[lab]
            
            # Each batch schedules labs independently
            for b in range(BATCHES_PER_DIVISION):
                batch_lab_sessions = []
                
                for day in days:
                    # Labs are 2-hour sessions, so we need consecutive slots
                    for s in range(len(slots) - 1):
                        # Skip if lunch interferes
                        if LUNCH_SLOT is not None and (s == LUNCH_SLOT or s + 1 == LUNCH_SLOT):
                            continue
                        
                        # Boolean for lab session starting at this time for this batch
                        lab_session = model.NewBoolVar(f'lab_{d}_b{b}_{lab}_{day}_{s}')
                        batch_lab_sessions.append(lab_session)
                        
                        # If lab session starts, occupy 2 consecutive slots for this batch only
                        model.Add(x[d][b][day][s] == lab_val).OnlyEnforceIf(lab_session)
                        model.Add(x[d][b][day][s+1] == lab_val).OnlyEnforceIf(lab_session)
                
                # Each batch must meet minimum lab frequency requirement
                model.Add(sum(batch_lab_sessions) == class_frequencies[lab])

    # --------------------
    # RESOURCE CONSTRAINTS
    # --------------------
    print("Adding resource constraints...")
    
    # Calculate available resources for labs
    lab_teacher_count = {}
    lab_room_count = {}
    for lab in labs:
        lab_teacher_count[lab] = sum(1 for classes in teacher_assignments.values() if lab in classes)
        lab_room_count[lab] = sum(1 for classes in room_assignments.values() if lab in classes)
        print(f"Lab {lab}: {lab_teacher_count[lab]} teachers, {lab_room_count[lab]} rooms available")

    # Resource constraints for each time slot
    for day in days:
        for s in slots:
            if s == LUNCH_SLOT:
                continue
                
            # Lab capacity constraints - each lab session uses one teacher and one room
            for lab in labs:
                lab_val = class_to_idx[lab]
                
                # Count concurrent lab sessions across all divisions and batches
                lab_usage_count = []
                for d in divisions:
                    for b in range(BATCHES_PER_DIVISION):
                        using_lab = model.NewBoolVar(f'using_{lab}_{d}_b{b}_{day}_{s}')
                        model.Add(x[d][b][day][s] == lab_val).OnlyEnforceIf(using_lab)
                        model.Add(x[d][b][day][s] != lab_val).OnlyEnforceIf(using_lab.Not())
                        lab_usage_count.append(using_lab)
                
                # Limit concurrent lab usage based on available teachers and rooms
                max_concurrent = min(lab_teacher_count[lab], lab_room_count[lab])
                model.Add(sum(lab_usage_count) <= max_concurrent)

    # --------------------
    # DIVISION-SPECIFIC TEACHER CONSTRAINTS (Inherent in assignment)
    # --------------------
    print("Division-specific teachers ensure no cross-division conflicts!")

    # Additional constraint: Ensure lecture synchronization within divisions
    print("Adding lecture synchronization constraints...")
    for d in divisions:
        for day in days:
            for s in slots:
                if s == LUNCH_SLOT:
                    continue
                
                # For each lecture, if any batch has it, all batches in the division must have it
                for lec in lectures:
                    lec_val = class_to_idx[lec]
                    
                    # Create boolean variables for each batch having this lecture
                    batch_has_lecture = []
                    for b in range(BATCHES_PER_DIVISION):
                        has_lec = model.NewBoolVar(f'div_{d}_b{b}_has_{lec}_{day}_{s}')
                        model.Add(x[d][b][day][s] == lec_val).OnlyEnforceIf(has_lec)
                        model.Add(x[d][b][day][s] != lec_val).OnlyEnforceIf(has_lec.Not())
                        batch_has_lecture.append(has_lec)
                    
                    # All batches must have the same status for this lecture
                    for b in range(1, BATCHES_PER_DIVISION):
                        model.Add(batch_has_lecture[0] == batch_has_lecture[b])

    # --------------------
    # Solve
    # --------------------
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 180.0
    solver.parameters.num_search_workers = 4
    
    print("\nStarting solver with division-specific teachers...")
    print(f"Problem size: {len(divisions)} divisions × {BATCHES_PER_DIVISION} batches × {len(days)} days × {len(slots)} slots")
    
    start_time = time.time()
    status = solver.Solve(model)
    solve_time = time.time() - start_time
    print(f"Solver finished in {solve_time:.2f} seconds.")

    # --------------------
    # Generate Results with Division-Specific Teachers
    # --------------------
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(f"\nSolution found! Status: {solver.StatusName(status)}")
        
        time_headers = [f'{start_hour+i}:00-{start_hour+i+1}:00' for i in slots]
        timetable_json = {}

        for d in divisions:
            print(f'\n=============== Division {d} ===============')
            
            # Display combined table with division-specific teachers
            table_data = []
            json_data = {}
            
            for day in days:
                row = [day]
                json_row = []
                
                for s in slots:
                    if s == LUNCH_SLOT:
                        row.append("LUNCH")
                        json_row.append({"type": "LUNCH"})
                        continue

                    # Get what each batch is doing
                    batch_activities = []
                    for b in range(BATCHES_PER_DIVISION):
                        val = solver.Value(x[d][b][day][s])
                        if val == -1:
                            batch_activities.append("Free")
                        else:
                            batch_activities.append(idx_to_class[val])
                    
                    # Summarize the slot with division-specific teacher information
                    unique_activities = set(batch_activities)
                    if len(unique_activities) == 1:
                        activity = batch_activities[0]
                        if activity == "Free":
                            row.append("Free")
                            json_row.append({"type": "Free"})
                        else:
                            teacher = division_class_to_teacher[d].get(activity, "No teacher assigned")
                            room = class_to_room.get(activity, "")
                            
                            row.append(f"[ALL] {activity}\\n{teacher}")
                            json_row.append({"type": "Lecture" if activity in lectures else "Lab", 
                                           "class": activity, "teacher": teacher, "room": room, "batches": "ALL"})
                    else:
                        # Mixed activities - show batch-wise breakdown with division-specific teachers
                        summary_parts = []
                        activities = []
                        
                        activity_count = {}
                        for i, activity in enumerate(batch_activities):
                            if activity not in activity_count:
                                activity_count[activity] = []
                            activity_count[activity].append(i + 1)  # Batch numbers start from 1
                        
                        for activity, batches in activity_count.items():
                            batch_str = ",".join(map(str, batches))
                            if activity == "Free":
                                summary_parts.append(f"B{batch_str}: Free")
                                activities.append({"batches": batch_str, "class": "Free"})
                            else:
                                teacher = division_class_to_teacher[d].get(activity, "No teacher assigned")
                                room = class_to_room.get(activity, "")
                                activity_type = "Lecture" if activity in lectures else "Lab"
                                
                                summary_parts.append(f"B{batch_str}: {activity}\\n{teacher}")
                                activities.append({"batches": batch_str, "class": activity, 
                                                 "teacher": teacher, "room": room, "type": activity_type})
                        
                        row.append("\\n".join(summary_parts))
                        json_row.append({"type": "Mixed", "activities": activities})
                
                table_data.append(row)
                json_data[day] = json_row
            
            # Display combined table
            df = pd.DataFrame(table_data, columns=['Day'] + time_headers)
            print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))
            
            timetable_json[d] = json_data

        # Save results
        with open("timetable_division_specific.json", "w") as f:
            json.dump(timetable_json, f, indent=4)
        print(f"\nTimetable saved to timetable_division_specific.json")
        
        # Print summary with division-specific teachers
        print(f"\n=== DIVISION-SPECIFIC TEACHER ASSIGNMENTS ===")
        for d in divisions:
            print(f"\nDivision {d} Teachers:")
            for subject in lectures + labs:
                if subject in division_class_to_teacher[d]:
                    teacher = division_class_to_teacher[d][subject]
                    print(f"  {subject}: {teacher}")

        print(f"\n=== SCHEDULING SUMMARY ===")
        for d in divisions:
            print(f"\nDivision {d}:")
            
            # Lecture summary with division-specific teachers
            for lec in lectures:
                lec_val = class_to_idx[lec]
                total_lecture_slots = 0
                teacher = division_class_to_teacher[d].get(lec, "No teacher assigned")
                for day in days:
                    for s in slots:
                        if solver.Value(x[d][0][day][s]) == lec_val:  # Check any batch (all should be same for lectures)
                            total_lecture_slots += 1
                print(f"  Lecture {lec}: {total_lecture_slots} hours/week (Required: {class_frequencies[lec]}) - Teacher: {teacher}")
            
            # Lab summary with teachers
            for lab in labs:
                lab_val = class_to_idx[lab]
                batch_lab_hours = [0] * BATCHES_PER_DIVISION
                teacher = division_class_to_teacher[d].get(lab, "No teacher assigned")
                
                for b in range(BATCHES_PER_DIVISION):
                    for day in days:
                        for s in slots:
                            if solver.Value(x[d][b][day][s]) == lab_val:
                                batch_lab_hours[b] += 1
                
                # Convert to sessions (each session is 2 hours)
                batch_sessions = [h // 2 for h in batch_lab_hours]
                print(f"  Lab {lab}: Sessions per batch {batch_sessions} (Required: {class_frequencies[lab]} each) - Teacher: {teacher}")

    else:
        print(f"\nNo solution found. Status: {solver.StatusName(status)}")
        print("The problem may be over-constrained.")

if __name__ == "__main__":
    main()

Successfully loaded all JSON configuration files.
Lunch break is scheduled for slot index 3 (12:00 - 13:00).
Planning timetable for 4 divisions, each with 4 batches
Available slots per day: 7, Total weekly slots: 35
Lectures: ['DSA', 'MA', 'GenAI', 'UHV', 'Stats', 'BSE'] (Total: 11 hours/week)
Labs: ['DSL', 'DEVL', 'MAL'] (Total per batch: 12 hours/week)
Note: Labs can accommodate 1/4 of class at a time
Teachers are now division-specific - each teacher teaches only one division
Adding lecture constraints...
Adding lab constraints...
Adding resource constraints...
Lab DSL: 12 teachers, 10 rooms available
Lab DEVL: 12 teachers, 10 rooms available
Lab MAL: 12 teachers, 10 rooms available
Division-specific teachers ensure no cross-division conflicts!
Adding lecture synchronization constraints...

Starting solver with division-specific teachers...
Problem size: 4 divisions × 4 batches × 5 days × 8 slots
Solver finished in 1.51 seconds.

Solution found! Status: OPTIMAL

=============== Divis